# Computer Vision — Part 3 · Practical Notebook


**Companion lab for the lecture: Object Detection (Sliding windows → YOLO → R-CNN family · IoU · NMS · mAP) → Face Recognition (Siamese networks · Triplet loss) → Neural Style Transfer (content & style cost · Gram matrices)**

---

This is the **applications** notebook — three of the most important things
classical and modern computer vision is *used* for in the real world.

**Practical philosophy.** We use the tools people actually deploy:
- For **object detection** we use **Ultralytics YOLOv8** (the modern YOLO) for
  inference and **torchvision's Faster R-CNN** for the two-stage comparison.
  Nobody implements YOLO or Faster R-CNN from scratch in a real job — they load
  pretrained weights and use them.
- We **hand-implement IoU and NMS** because they're short, used everywhere, and
  the single best way to *understand* what detection libraries do under the hood.
- For **face recognition** we build a working **Siamese network with triplet
  loss** on a small image-pairs problem so you see the lecture's algorithm
  actually learn. We also show the production approach: use a pretrained
  embedder.
- For **style transfer** we implement the **Gatys et al.** algorithm from the
  lecture — pretrained VGG, content + Gram-matrix style loss, optimize the
  pixels of an image. It's about 50 lines of real code.

> **How to run this:** Google Colab with a GPU is strongly recommended
> (`Runtime → Change runtime type → GPU`) — particularly for Sections 10 and 12
> which actually train/optimize. CPU works for everything else but Section 12
> will take several minutes per run.


## 1 · Setup & device

Standard imports for this part. We'll add Ultralytics later only when we need
it (Section 4) to keep the early sections fast.


In [ ]:
# Uncomment on a fresh environment:
# !pip install torch torchvision ultralytics matplotlib pillow --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models, transforms
from torchvision.ops import nms as tv_nms, box_iou as tv_box_iou
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
import io, urllib.request, os

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch :", torch.__version__)
print("device  :", device)


## 2 · Intersection over Union (IoU) — from scratch

> *Lecture recap:* **IoU = area of intersection ÷ area of union** of two
> bounding boxes. It's how we judge whether a predicted box matches a
> ground-truth box. By convention a prediction is considered **correct if
> IoU ≥ 0.5** (and the class is right). Perfect overlap → IoU = 1; no overlap →
> IoU = 0.

This is a 10-line function that appears inside *every* object detection system —
training (assigning anchors to ground truth), evaluation (mAP), and inference
(NMS, next section). Let's write it.

**Box convention.** We use **`[x1, y1, x2, y2]`** — the top-left and
bottom-right corners. This is the standard in `torchvision`, `albumentations`,
and most modern code. (The lecture's `(bx, by, bh, bw)` — centre + size — is
YOLO's *output* format; we'll convert in Section 6.)


In [ ]:
def iou(boxA, boxB):
    """
    IoU of two boxes given as [x1, y1, x2, y2].
    Returns a scalar in [0, 1].
    """
    # 1) coordinates of the intersection rectangle
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    # 2) intersection area -- clip negatives to 0 (no overlap case)
    inter_w = max(0.0, xB - xA)
    inter_h = max(0.0, yB - yA)
    inter = inter_w * inter_h

    # 3) areas of each box, then union = A + B - intersection
    areaA = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    areaB = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union = areaA + areaB - inter

    return inter / union if union > 0 else 0.0

# Visual sanity check.
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
cases = [
    ("perfect overlap",   [50, 50, 150, 150], [50, 50, 150, 150]),
    ("partial overlap",   [50, 50, 150, 150], [100, 80, 200, 180]),
    ("no overlap",        [50, 50, 100, 100], [120, 120, 200, 200]),
]
for ax, (title, a, b) in zip(axes, cases):
    score = iou(a, b)
    ax.add_patch(mpatches.Rectangle((a[0], a[1]), a[2]-a[0], a[3]-a[1],
                                    fill=False, edgecolor="green", lw=2, label="A"))
    ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                    fill=False, edgecolor="red",   lw=2, label="B"))
    ax.set_xlim(0, 250); ax.set_ylim(250, 0)        # flip y so origin is top-left
    ax.set_aspect("equal")
    ax.set_title(f"{title}\nIoU = {score:.3f}")
plt.tight_layout(); plt.show()


Now confirm our implementation matches `torchvision.ops.box_iou` — that's
the function every PyTorch-based detector actually uses internally:


In [ ]:
# torchvision's batched version: feed two tensors of boxes, get an (N x M) matrix.
A = torch.tensor([[50, 50, 150, 150],
                  [50, 50, 150, 150],
                  [50, 50, 100, 100]], dtype=torch.float32)
B = torch.tensor([[50, 50, 150, 150],
                  [100, 80, 200, 180],
                  [120, 120, 200, 200]], dtype=torch.float32)

ours = [iou(a.tolist(), b.tolist()) for a, b in zip(A, B)]
tv   = tv_box_iou(A, B).diag().tolist()    # diagonal = each A vs its paired B
print("ours          :", [round(v, 4) for v in ours])
print("tv.ops.box_iou:", [round(v, 4) for v in tv])
print("match         :", all(abs(a - b) < 1e-6 for a, b in zip(ours, tv)))


### TODO 2.1
The lecture mentions "by convention an answer is correct if IoU ≥ 0.5". Write a
function `is_correct(pred_box, gt_box, threshold=0.5)` that returns `True` only
if both **IoU exceeds the threshold** *and* the class labels match. The function
should accept `(box, class_id)` pairs.

Then test it on three cases the instructor cell will define for you.


In [ ]:
# TODO 2.1
def is_correct(pred, gt, threshold=0.5):
    """
    pred = (box, class_id), gt = (box, class_id)
    Returns True iff the IoU is at least `threshold` AND the classes match.
    """
    pred_box, pred_cls = pred
    gt_box,   gt_cls   = gt
    # compute IoU and compare both conditions
    return ???

# Test cases
tests = [
    # (pred, gt, expected_result, why)
    ((( 50,  50, 150, 150), 1), (( 60,  55, 155, 160), 1), True,  "good IoU, same class"),
    ((( 50,  50, 150, 150), 1), (( 60,  55, 155, 160), 2), False, "good IoU, WRONG class"),
    (((  0,   0,  40,  40), 1), ((100, 100, 200, 200), 1), False, "right class, no overlap"),
]
for pred, gt, expected, why in tests:
    got = is_correct(pred, gt)
    print(f"  {why:38s} -> got {got},  expected {expected},  {'OK' if got == expected else 'FAIL'}")


## 3 · Non-Max Suppression (NMS) — from scratch

> *Lecture recap:* a detector typically fires multiple times around the same
> object — many grid cells/anchors all think they see the same car. **Non-Max
> Suppression** cleans up these duplicates:
> 1. **Drop** any box with confidence below a threshold (e.g. `score < 0.6`).
> 2. **Pick** the highest-scoring remaining box, **keep it**, output it.
> 3. **Drop** every other remaining box that overlaps it heavily
>    (IoU ≥ some threshold, e.g. 0.5).
> 4. Repeat steps 2–3 until no boxes remain.
> For multi-class output, run NMS *independently per class*.

This runs at the end of every detector (YOLO, Faster R-CNN, RetinaNet, DETR's
variants…). Let's implement it.


In [ ]:
def nms(boxes, scores, iou_thresh=0.5, score_thresh=0.6):
    """
    boxes:  list of [x1, y1, x2, y2]
    scores: list of confidences (parallel to boxes)
    Returns the indices of boxes kept after NMS.
    """
    # 1) drop low-score boxes immediately
    keep_candidates = [i for i, s in enumerate(scores) if s >= score_thresh]
    # 2) sort the survivors by score, highest first
    keep_candidates.sort(key=lambda i: scores[i], reverse=True)

    kept = []
    while keep_candidates:
        # 3) take the top one and commit to keeping it
        best = keep_candidates.pop(0)
        kept.append(best)
        # 4) drop everything that overlaps it heavily
        keep_candidates = [
            i for i in keep_candidates
            if iou(boxes[best], boxes[i]) < iou_thresh
        ]
    return kept

# Let's simulate the lecture's scenario: many duplicate detections of two cars.
boxes = [
    # cluster around car #1
    [50,  60, 150, 160],  [55,  65, 158, 165], [48,  58, 145, 155],  [60,  70, 162, 170],
    # cluster around car #2
    [300, 80, 420, 200], [305, 85, 422, 205], [310, 90, 430, 210],
    # an isolated low-confidence false positive
    [200, 200, 260, 260],
]
scores = [0.95, 0.91, 0.83, 0.78,    0.92, 0.88, 0.81,    0.30]

kept = nms(boxes, scores, iou_thresh=0.5, score_thresh=0.6)
print("input  : 8 detections")
print("kept   :", len(kept), "->", kept)
print("scores :", [scores[i] for i in kept], " <- the strongest of each cluster")


In [ ]:
# Visualize before vs after NMS.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, title, indices in [(ax1, "BEFORE NMS (8 boxes)", range(len(boxes))),
                            (ax2, "AFTER NMS (clean)", kept)]:
    for i in indices:
        b, s = boxes[i], scores[i]
        ax.add_patch(mpatches.Rectangle((b[0], b[1]), b[2]-b[0], b[3]-b[1],
                                        fill=False, edgecolor="red", lw=2))
        ax.text(b[0], b[1]-4, f"{s:.2f}", color="red", fontsize=9)
    ax.set_xlim(0, 500); ax.set_ylim(280, 0); ax.set_aspect("equal")
    ax.set_title(title)
plt.tight_layout(); plt.show()


And confirm against `torchvision.ops.nms` — the production version that's
GPU-accelerated and used inside every PyTorch detector:


In [ ]:
# torchvision needs tensors. Note it doesn't apply a score threshold itself
# (you pre-filter), so we mimic our two-step process exactly.
boxes_t  = torch.tensor(boxes, dtype=torch.float32)
scores_t = torch.tensor(scores, dtype=torch.float32)

mask = scores_t >= 0.6
filt_boxes  = boxes_t[mask]
filt_scores = scores_t[mask]
orig_idx    = mask.nonzero(as_tuple=True)[0]      # map filtered idx -> original

tv_kept_local = tv_nms(filt_boxes, filt_scores, iou_threshold=0.5)
tv_kept = orig_idx[tv_kept_local].tolist()

print("ours :", sorted(kept))
print("tv   :", sorted(tv_kept))
print("match:", sorted(kept) == sorted(tv_kept))


### TODO 3.1
Sweep the **IoU threshold** to see its effect. Run NMS with `iou_thresh = 0.3`,
`0.5`, and `0.9` on the same `boxes`/`scores` as above. Print how many boxes
each one keeps and explain (one line each):
- Why does a *low* IoU threshold keep *fewer* boxes?
- Why does a *high* IoU threshold keep *more* boxes?
- What disaster could a too-high IoU threshold cause in practice?


In [ ]:
# TODO 3.1
for thresh in [0.3, 0.5, 0.9]:
    kept_i = nms(boxes, scores, iou_thresh=thresh, score_thresh=0.6)
    print(f"iou_thresh = {thresh}  ->  kept {len(kept_i)} boxes  (indices {kept_i})")

# Explain in comments:
#  - Why does LOW iou_thresh keep FEWER boxes?
#  - Why does HIGH iou_thresh keep MORE boxes?
#  - What real-world failure can a too-high iou_thresh cause?


## 4 · YOLOv8 — real-world object detection inference

> *Lecture recap:* YOLO ("**You Only Look Once**") divides the image into a
> grid; each cell predicts `pc` (objectness), bounding box (`bx, by, bh, bw`),
> and class scores. It outputs all detections in **a single forward pass** —
> hence fast and real-time-capable. With **anchor boxes**, each cell can detect
> multiple objects of different shapes.

The original 2016 YOLO has evolved through v2, v3, v4, v5, v7, v8, v9, v10,
v11. **YOLOv8** is the current production default. Loading and running it is
**three lines**.


In [ ]:
# First time only: this downloads yolov8n.pt (~6 MB).
# (yolov8n = 'nano', the smallest+fastest variant. Bigger: s, m, l, x.)
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
print("YOLOv8 nano loaded.")
print("Number of classes:", len(model.names))   # COCO has 80 classes
print("Some classes:", list(model.names.values())[:10], "...")


In [ ]:
# Run on a sample image. We grab a known-good street scene from the COCO
# project (cars + people, perfect for showing detection).
SAMPLE_URL = "https://ultralytics.com/images/bus.jpg"
SAMPLE_PATH = "bus.jpg"
if not os.path.exists(SAMPLE_PATH):
    urllib.request.urlretrieve(SAMPLE_URL, SAMPLE_PATH)

# This is the production one-liner. verbose=False keeps logs quiet.
results = model(SAMPLE_PATH, verbose=False)
r = results[0]                   # results is a list (one item per input image)

# Inspect what came back.
print("detections    :", len(r.boxes))
print("boxes (xyxy)  : tensor of shape", r.boxes.xyxy.shape)
print("confidences   : tensor of shape", r.boxes.conf.shape)
print("class ids     : tensor of shape", r.boxes.cls.shape)

# Print a clean detection table.
print(f"\n{'class':12s} | conf  | box [x1, y1, x2, y2]")
print("-" * 60)
for box, conf, cls in zip(r.boxes.xyxy, r.boxes.conf, r.boxes.cls):
    name = model.names[int(cls)]
    x1, y1, x2, y2 = [int(v) for v in box]
    print(f"{name:12s} | {conf:.2f}  | [{x1}, {y1}, {x2}, {y2}]")


In [ ]:
# Visualize. Ultralytics ships a .plot() that draws boxes on the image --
# that's what you'd use in real code; we'll show it AND also draw with matplotlib
# so you see how to do it yourself.
annotated = r.plot()                       # numpy array, BGR order (OpenCV-style)
annotated_rgb = annotated[:, :, ::-1]      # convert BGR -> RGB for matplotlib

img_rgb = np.array(Image.open(SAMPLE_PATH).convert("RGB"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
ax1.imshow(img_rgb);       ax1.set_title("input"); ax1.axis("off")
ax2.imshow(annotated_rgb); ax2.set_title("YOLOv8 detections"); ax2.axis("off")
plt.tight_layout(); plt.show()


### TODO 4.1
Change the YOLOv8 **confidence threshold** to see how it affects the
predictions. Run the model with `conf = 0.05`, `0.25`, and `0.7` on the same
image and report how many detections each setting produces. Then answer:
- What kind of objects appear only with the lower threshold?
- For a *safety-critical* application (say, autonomous-vehicle pedestrian
  detection), would you tune `conf` higher or lower than the default?


In [ ]:
# TODO 4.1 -- sweep the confidence threshold.
for conf in [0.05, 0.25, 0.70]:
    res = model(SAMPLE_PATH, conf=conf, verbose=False)[0]
    print(f"conf >= {conf:.2f}  ->  {len(res.boxes):3d} detections")

# Answers (in comments):
#  - What kind of objects appear ONLY with low confidence?
#  - For a safety-critical app (pedestrian detection), tune higher or lower? Why?


## 5 · Fine-tuning YOLOv8 on a custom dataset

> *Lecture recap (Part 2):* you rarely train from scratch. Take a model
> pretrained on a big dataset, **replace** or **adapt** the final layers, then
> train on your smaller, domain-specific data. For YOLO, the same idea applies:
> the COCO-pretrained backbone already knows "what edges, shapes, and textures
> look like"; we adapt it to detect *new classes* with very little data.

This is the most common task in real applied CV: someone gives you a small
dataset of *their* objects (defects on a factory line, animals on a wildlife
camera, products on a shelf) and asks for a detector. The answer is almost
always **fine-tune YOLO on it**, not train from scratch.

We'll fine-tune on the **African Wildlife** dataset (buffalo, elephant, rhino,
zebra) — a small (~100 MB, 1052 train + 225 val images) dataset that Ultralytics
auto-downloads. None of the four animals are standard COCO detection classes,
so this is a genuine new-domain fine-tune.


In [ ]:
# Confirm we have the model and that GPU is available (training is much faster).
print("device                :", device)
if device.type == "cuda":
    print("GPU                   :", torch.cuda.get_device_name(0))
else:
    print("WARNING: training on CPU will be slow. ~5 min on GPU, ~20+ min on CPU.")

# Reload a fresh YOLOv8 nano with COCO weights -- this is our starting point.
# (model from Section 4 is the same; we make a fresh one for clarity.)
ft_model = YOLO("yolov8n.pt")
print("starting from COCO-pretrained YOLOv8 nano")


### 5.1 — The dataset

Ultralytics references datasets by short YAML configs. Calling `model.train(
data="african-wildlife.yaml", ...)` triggers the dataset's auto-download (only
the first time) and then trains. The YAML simply specifies where the train/val
images live and the class names.


In [ ]:
# Peek at the dataset config so the format isn't mysterious.
import ultralytics
from pathlib import Path
yaml_path = Path(ultralytics.__file__).parent / "cfg" / "datasets" / "african-wildlife.yaml"
print(yaml_path.read_text())


Format to internalize: this is **the** YOLO dataset format. To fine-tune
on **your own** data later, you write a YAML exactly like this and put your
images in `images/train` / `images/val` with matching label files in
`labels/train` / `labels/val`. Each label file has one line per object:
`class_id  x_center  y_center  width  height` — all normalized to 0–1.


### 5.2 — Fine-tune

`model.train(...)` runs the full training loop: data loading, augmentation,
forward/backward passes, validation, checkpointing, mAP computation. It writes
everything to `runs/detect/<name>/` so you can inspect logs, plots, and saved
weights after training.

For a **fast lab demo** we use small input size (`imgsz=320`), few epochs
(`epochs=5`), and the nano model. In a real project you'd push imgsz to 640
and epochs to 50-100 — but the *workflow* is identical.


In [ ]:
# The actual fine-tune. This is the entire training loop.
# On a Colab T4 GPU this takes ~3-5 minutes; on CPU it's ~20+ minutes.
results = ft_model.train(
    data="african-wildlife.yaml",   # auto-downloads on first run (~100 MB)
    epochs=5,                       # bump to 50+ for real projects
    imgsz=320,                      # smaller = faster lab demo; 640 is standard
    batch=16,
    name="wildlife_finetune",       # output folder: runs/detect/wildlife_finetune
    verbose=False,                  # quieter logs for notebook display
    plots=True,                     # save loss/mAP plots automatically
    patience=0,                     # disable early stopping for predictable timing
)
print("\nDone. Best weights saved at runs/detect/wildlife_finetune/weights/best.pt")


### 5.3 — Inspect what training produced

Ultralytics writes a folder of artifacts: best/last weights, loss curves,
example validation predictions, a confusion matrix, and a CSV of per-epoch
metrics. **Always look at these** — that's the habit that catches data bugs.


In [ ]:
# List what landed in the run directory.
run_dir = Path("runs/detect/wildlife_finetune")
print(f"run directory contents ({run_dir}):")
for f in sorted(run_dir.iterdir()):
    if f.is_file():
        print(f"  {f.name:30s}  {f.stat().st_size // 1024:>6d} KB")
    else:
        print(f"  {f.name}/")


In [ ]:
# Show the training plots Ultralytics generated automatically.
results_png = run_dir / "results.png"
if results_png.exists():
    from PIL import Image as _PIL
    plt.figure(figsize=(14, 5))
    plt.imshow(_PIL.open(results_png))
    plt.title("training curves (loss + metrics over epochs)")
    plt.axis("off"); plt.show()
    print("Read these like in Part 2: training loss should fall, val mAP should rise.")
else:
    print("results.png not yet present -- training may still be writing.")


In [ ]:
# Read the per-epoch metrics from results.csv -- this is what you'd parse
# programmatically in a real pipeline (e.g. to decide when to stop training).
import csv
metrics_csv = run_dir / "results.csv"
if metrics_csv.exists():
    with open(metrics_csv) as f:
        rows = list(csv.DictReader(f))
    # Keep only the most informative columns and strip whitespace from keys.
    rows = [{k.strip(): v for k, v in r.items()} for r in rows]
    interesting = ["epoch", "train/box_loss", "train/cls_loss",
                   "metrics/precision(B)", "metrics/recall(B)",
                   "metrics/mAP50(B)", "metrics/mAP50-95(B)"]
    print(f"{'epoch':>6s}  {'box_loss':>10s}  {'cls_loss':>10s}  "
          f"{'precision':>10s}  {'recall':>10s}  {'mAP50':>8s}  {'mAP50-95':>9s}")
    print("-" * 78)
    for r in rows:
        vals = [r.get(k, "?") for k in interesting]
        e = int(float(vals[0]))
        nums = [f"{float(v):>10.4f}" if v not in ("", "?") else f"{'?':>10s}" for v in vals[1:]]
        print(f"{e:>6d}  {nums[0]}  {nums[1]}  {nums[2]}  {nums[3]}  {nums[4][-8:]}  {nums[5][-9:]}")
    print("\nWatch mAP50 climb across epochs -> the fine-tune is working.")


### 5.4 — Use the fine-tuned model

After training, `ft_model` is updated *in place* — it now predicts the new
classes. Or you can reload from the saved weights file (`best.pt`). Either way,
inference is the same one-liner as before.


In [ ]:
# Validate on a held-out val image to see the new model's behaviour.
val_dir = run_dir / "val_batch0_pred.jpg"   # Ultralytics auto-saves a labeled val batch
if val_dir.exists():
    plt.figure(figsize=(10, 10))
    plt.imshow(Image.open(val_dir))
    plt.title("validation batch — model's predictions on unseen images")
    plt.axis("off"); plt.show()
else:
    print("val batch image not found at", val_dir)


In [ ]:
# Sanity check: the fine-tuned model now knows the four wildlife classes.
print("fine-tuned model classes:", ft_model.names)
print("(Compare to the original COCO classes from Section 4 -- completely different.)")


### TODO 5.1
You just ran an end-to-end fine-tune. Now reason about three real fine-tuning
decisions you'd face on your own project:

1. **Compare the fine-tuned model's mAP50 (above) to what you'd get with the
   *un-fine-tuned* `yolov8n.pt` on this wildlife data.** What would the
   un-fine-tuned model's mAP50 on buffalo/elephant/rhino/zebra be? Why?
   (Hint: look at COCO's class list and look for these animals.)

2. **You have 100 labeled images of a new defect type.** Which would you do:
   train from scratch, fine-tune YOLOv8, or use the pretrained model as-is
   without training? Why?

3. **Your fine-tuned model achieves 92% mAP50 on the val set but only 60% in
   production.** Name two likely causes and one thing you'd check first.


In [ ]:
# TODO 5.1 -- reason about fine-tuning decisions.

# 1) Un-fine-tuned YOLOv8 on buffalo/elephant/rhino/zebra: expected mAP50?
#    Look at the COCO class list (the lecture's "80 classes") -- which of these
#    four animals are in it?
#    Your answer:
#      -> ...

# 2) 100 labeled images of a new defect type. Best approach?
#    Your answer:
#      -> ...

# 3) val mAP50 = 92%, production = 60%. Two likely causes, one thing to check?
#    Your answer:
#      -> ...


## 7 · Faster R-CNN — the two-stage detector

> *Lecture recap:* the R-CNN family is the **two-stage** alternative to YOLO:
> - **R-CNN** (2014): Selective Search → ~2000 region proposals → run a CNN
>   on each (slow, ~2000 forward passes).
> - **Fast R-CNN**: run CNN **once** on the whole image → get a feature map →
>   use **RoI Pooling** to extract a fixed-size feature for each proposal →
>   classify. Much faster.
> - **Faster R-CNN**: replace Selective Search with a **Region Proposal Network
>   (RPN)** that *also* uses the shared feature map. End-to-end trainable, and
>   the standard two-stage detector.

The honest practical reality: **nobody implements Faster R-CNN from scratch**.
`torchvision.models.detection` ships pretrained Faster R-CNN with one line.
Let's load it and compare to YOLO on the same image.


In [ ]:
# Pretrained Faster R-CNN with a ResNet-50 + FPN backbone, COCO weights.
# First time: ~160 MB download.
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
)

w = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
frcnn = fasterrcnn_resnet50_fpn(weights=w).eval().to(device)
COCO_CLASSES = w.meta["categories"]
print("Faster R-CNN loaded. COCO has", len(COCO_CLASSES), "classes")


In [ ]:
# Run Faster R-CNN on the same image YOLO used.
img = Image.open(SAMPLE_PATH).convert("RGB")
img_tensor = transforms.ToTensor()(img).to(device)        # the only preprocessing needed

with torch.no_grad():
    pred = frcnn([img_tensor])[0]    # input is a list (batch); output mirrors

# pred is a dict with 'boxes', 'scores', 'labels'.
SCORE_TH = 0.6
keep = pred["scores"] >= SCORE_TH
boxes  = pred["boxes"][keep].cpu()
scores = pred["scores"][keep].cpu()
labels = pred["labels"][keep].cpu()

print(f"Faster R-CNN detections (score >= {SCORE_TH}):", len(boxes))
print(f"\n{'class':12s} | conf  | box")
print("-" * 60)
for b, s, l in zip(boxes, scores, labels):
    x1, y1, x2, y2 = [int(v) for v in b]
    print(f"{COCO_CLASSES[l]:12s} | {s:.2f}  | [{x1}, {y1}, {x2}, {y2}]")


In [ ]:
# Side-by-side: YOLO output (from Section 4) vs Faster R-CNN.
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Replot YOLO using Ultralytics' renderer.
yolo_res = model(SAMPLE_PATH, verbose=False)[0]
axes[0].imshow(yolo_res.plot()[:, :, ::-1])    # BGR->RGB
axes[0].set_title(f"YOLOv8 nano ({len(yolo_res.boxes)} detections)")
axes[0].axis("off")

# Plot Faster R-CNN with matplotlib.
axes[1].imshow(np.array(img))
for b, s, l in zip(boxes, scores, labels):
    x1, y1, x2, y2 = b.tolist()
    axes[1].add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                          fill=False, edgecolor="lime", lw=2))
    axes[1].text(x1, y1 - 4, f"{COCO_CLASSES[l]} {s:.2f}",
                  color="lime", fontsize=9,
                  bbox=dict(facecolor="black", alpha=0.5, pad=1, edgecolor="none"))
axes[1].set_title(f"Faster R-CNN ResNet50 FPN ({len(boxes)} detections)")
axes[1].axis("off")

plt.tight_layout(); plt.show()
print("\nYOLO and Faster R-CNN usually find roughly the same objects.")
print("Faster R-CNN tends to be slightly more accurate; YOLO is much faster.")


### TODO 7.1
**Time them.** Benchmark a single forward pass of YOLOv8 vs Faster R-CNN on the
same image, average over a few runs. Compute the speed ratio and connect it to
the lecture's "YOLO is fast enough for real-time" claim.

(If on CPU, the gap will be enormous; on GPU it's smaller but still real.)


In [ ]:
# TODO 7.1 -- timing comparison.
import time

# Warm up once (first call sets up CUDA kernels and caches).
_ = model(SAMPLE_PATH, verbose=False)
with torch.no_grad():
    _ = frcnn([img_tensor])

N_RUNS = 5

# Time YOLOv8.
t0 = time.time()
for _ in range(N_RUNS):
    _ = model(SAMPLE_PATH, verbose=False)
yolo_time = (time.time() - t0) / N_RUNS

# Time Faster R-CNN.
t0 = time.time()
for _ in range(N_RUNS):
    with torch.no_grad():
        _ = frcnn([img_tensor])
frcnn_time = (time.time() - t0) / N_RUNS

ratio = ???
print(f"YOLOv8 nano   : {yolo_time*1000:6.1f} ms/image   ({1/yolo_time:5.1f} FPS)")
print(f"Faster R-CNN  : {frcnn_time*1000:6.1f} ms/image  ({1/frcnn_time:5.1f} FPS)")
print(f"YOLOv8 is ~{ratio:.1f}x faster on this hardware.")

# Comment: at this speed, can YOLOv8 process a 30-fps video in real time?
#         Can Faster R-CNN?


## 8 · RF-DETR — the transformer-based detector

We've now seen two archetypes:
- **YOLO** (Section 4) — *one-stage*: a single CNN predicts boxes + classes
  directly from a grid output. Fast, simple.
- **Faster R-CNN** (Section 7) — *two-stage*: an RPN proposes regions, then a
  second head classifies them. Slightly more accurate, ~10× slower.

There's a third, newer archetype: **transformer-based detection**. Instead of
grids + anchors + NMS, the detector emits a **fixed set of predictions** and
uses self-attention (the same idea as Part 2's ViT) to reason about objects
*globally* across the image. The breakthrough paper was **DETR** (Facebook
AI, 2020), but it was slow. **RF-DETR** (Roboflow, 2025) is a real-time
transformer detector that matches YOLO's speed *and* beats it on accuracy at
similar size — a practical signal that transformers have arrived in detection.

We'll briefly run RF-DETR for inference on the same image we used for YOLO and
Faster R-CNN, so you can compare three detector families side-by-side using
identical code patterns.


In [ ]:
# Uncomment on a fresh environment:
# !pip install rfdetr supervision --quiet
from rfdetr import RFDETRNano
import supervision as sv

# RFDETRNano matches yolov8n's role: smallest, fastest variant.
# First time: downloads pretrained COCO weights (~30 MB).
rfdetr = RFDETRNano()
print("RF-DETR Nano loaded.")
print("Number of classes:", len(rfdetr.class_names))
print("First 10 classes :", rfdetr.class_names[:10], "...")


In [ ]:
# Inference is one line. RF-DETR uses confidence threshold directly (like YOLO).
# The output is a `supervision.Detections` object -- a unified format that works
# with detectors from many libraries.
rf_detections = rfdetr.predict(SAMPLE_PATH, threshold=0.5)

print(f"detections : {len(rf_detections)}")
print(f"boxes      : shape {rf_detections.xyxy.shape}, format [x1, y1, x2, y2]")
print(f"confidences: shape {rf_detections.confidence.shape}")
print(f"class ids  : shape {rf_detections.class_id.shape}")

# Clean detection table.
print(f"\n{'class':12s} | conf  | box")
print("-" * 60)
for box, conf, cls in zip(rf_detections.xyxy, rf_detections.confidence, rf_detections.class_id):
    name = rfdetr.class_names[int(cls)]
    x1, y1, x2, y2 = [int(v) for v in box]
    print(f"{name:12s} | {conf:.2f}  | [{x1}, {y1}, {x2}, {y2}]")


In [ ]:
# Visualize RF-DETR's predictions using supervision's annotators.
import numpy as np
img_array = np.array(Image.open(SAMPLE_PATH).convert("RGB"))

box_annotator   = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator(text_position=sv.Position.TOP_LEFT)
labels = [f"{rfdetr.class_names[int(c)]} {p:.2f}"
          for c, p in zip(rf_detections.class_id, rf_detections.confidence)]
annotated = box_annotator.annotate(img_array.copy(), rf_detections)
annotated = label_annotator.annotate(annotated, rf_detections, labels=labels)

plt.figure(figsize=(10, 7))
plt.imshow(annotated); plt.title(f"RF-DETR Nano ({len(rf_detections)} detections)")
plt.axis("off"); plt.show()


### Three detectors on one image — the final comparison

Now let's put all three families side-by-side. This is the kind of comparison
plot you'd run when *choosing* a detector for a real project.


In [ ]:
# YOLOv8 again (from Section 4) for the three-way comparison.
yolo_r = model(SAMPLE_PATH, verbose=False)[0]

# Faster R-CNN again (from Section 7).
img_t = transforms.ToTensor()(Image.open(SAMPLE_PATH).convert("RGB")).to(device)
with torch.no_grad():
    frcnn_pred = frcnn([img_t])[0]
frcnn_keep = frcnn_pred["scores"] >= 0.5
frcnn_boxes  = frcnn_pred["boxes"][frcnn_keep].cpu()
frcnn_labels = frcnn_pred["labels"][frcnn_keep].cpu()
frcnn_scores = frcnn_pred["scores"][frcnn_keep].cpu()

# Three-panel figure.
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Panel 1: YOLOv8
axes[0].imshow(yolo_r.plot()[:, :, ::-1])
axes[0].set_title(f"YOLOv8 nano (one-stage)\n{len(yolo_r.boxes)} detections")
axes[0].axis("off")

# Panel 2: Faster R-CNN
axes[1].imshow(img_array)
for b, s, l in zip(frcnn_boxes, frcnn_scores, frcnn_labels):
    x1, y1, x2, y2 = b.tolist()
    axes[1].add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                         fill=False, edgecolor="lime", lw=2))
    axes[1].text(x1, y1-4, f"{COCO_CLASSES[l]} {s:.2f}",
                 color="lime", fontsize=8,
                 bbox=dict(facecolor="black", alpha=0.5, pad=1, edgecolor="none"))
axes[1].set_title(f"Faster R-CNN (two-stage)\n{len(frcnn_boxes)} detections")
axes[1].axis("off")

# Panel 3: RF-DETR
axes[2].imshow(annotated)
axes[2].set_title(f"RF-DETR (transformer)\n{len(rf_detections)} detections")
axes[2].axis("off")

plt.suptitle("Three detection families on the same image", fontsize=13)
plt.tight_layout(); plt.show()


### TODO 8.1
Pick the right detector for each scenario and justify in **one line** using
the speed/accuracy/data trade-offs from Sections 4, 7, and 8:

1. **Drone autopilot** detecting obstacles, runs on a small onboard computer
   at 30 FPS.
2. **Medical research project** — accurately count tumour cells in 500
   microscope images (offline, accuracy is everything).
3. **New startup** building a domain-specific detector — they have ~80,000
   labeled images and want best-in-class accuracy, deploying on a cloud GPU
   server.


In [ ]:
# TODO 8.1
# Pick: YOLOv8 / Faster R-CNN / RF-DETR for each scenario. One-line justification.
#
# 1) Drone autopilot, onboard compute, 30 FPS, obstacle detection:
#    -> ...
#
# 2) Medical research, count cells in 500 images, offline, max accuracy:
#    -> ...
#
# 3) Startup with 80k labeled images, cloud GPU, best accuracy:
#    -> ...


## 9 · mean Average Precision (mAP) — the standard detection metric

> *Lecture recap:* a detector outputs many boxes per image with confidence
> scores. To evaluate it, for each predicted box we check: **is it correct?**
> (class right AND IoU ≥ 0.5 with an unmatched GT). **Each GT can only be
> matched once** — extra overlapping predictions become false positives.
> Sliding the confidence threshold from high to low traces out a
> **Precision–Recall curve**. **Average Precision (AP)** is roughly the area
> under that curve, computed per class; **mAP** is the mean of AP across all
> classes.

Real codebases use `pycocotools` or `torchmetrics`; we'll compute AP **from
scratch** on a tiny toy set so the four steps (match → sort by confidence →
precision-recall curve → area under it) are unambiguous.


In [ ]:
# Toy example: one image, three ground-truth objects, six predictions.
# All boxes are class 0 for simplicity (so this is per-class AP, no class loop).
ground_truths = [
    [ 50,  50, 150, 150],   # GT 0
    [200,  50, 300, 150],   # GT 1
    [100, 200, 200, 300],   # GT 2
]
# (box, confidence)
predictions = [
    ([ 55,  55, 145, 145], 0.95),   # great match with GT 0
    ([205,  55, 305, 155], 0.90),   # great match with GT 1
    ([ 60,  60, 140, 140], 0.88),   # ALSO overlaps GT 0 -> duplicate FP
    ([105, 205, 195, 295], 0.70),   # great match with GT 2
    ([400, 400, 450, 450], 0.55),   # FP, nowhere near any GT
    ([110, 215, 180, 280], 0.40),   # also overlaps GT 2 -> duplicate FP
]

def compute_ap(predictions, ground_truths, iou_thresh=0.5):
    """
    Compute Average Precision for ONE class on ONE image.
    Returns: precision array, recall array, AP scalar.
    """
    n_gt = len(ground_truths)
    # Sort predictions by confidence, descending -- this is THE key step.
    preds_sorted = sorted(predictions, key=lambda p: p[1], reverse=True)

    matched_gt = set()                           # which GT indices already matched
    tp = np.zeros(len(preds_sorted))             # per-prediction: 1 if true positive
    fp = np.zeros(len(preds_sorted))             # per-prediction: 1 if false positive

    for i, (pred_box, _) in enumerate(preds_sorted):
        # Find the best-matching GT (highest IoU, above threshold, not yet matched).
        best_iou, best_j = 0.0, -1
        for j, gt_box in enumerate(ground_truths):
            if j in matched_gt:
                continue
            cur = iou(pred_box, gt_box)
            if cur > best_iou:
                best_iou, best_j = cur, j
        if best_iou >= iou_thresh:
            tp[i] = 1
            matched_gt.add(best_j)                # this GT is now claimed -- no doubles
        else:
            fp[i] = 1

    # Cumulative TP and FP as we walk down the sorted list.
    cum_tp = np.cumsum(tp); cum_fp = np.cumsum(fp)
    precision = cum_tp / (cum_tp + cum_fp + 1e-9)
    recall    = cum_tp / n_gt

    # AP = area under the precision-recall curve (11-point or all-points; we use
    # the simple all-points trapezoidal version here).
    # Prepend (recall=0, precision=1) and append (recall=last, precision=0)
    # for a clean integration.
    r = np.concatenate([[0.0], recall, [recall[-1]]])
    p = np.concatenate([[1.0], precision, [0.0]])
    # Make precision non-increasing as recall grows (standard PR-curve smoothing).
    for k in range(len(p) - 1, 0, -1):
        p[k - 1] = max(p[k - 1], p[k])
    # np.trapz was renamed to np.trapezoid in NumPy 2.0; support both versions.
    try:
        ap = np.trapezoid(p, r)
    except AttributeError:
        ap = np.trapz(p, r)
    return precision, recall, ap, preds_sorted, tp, fp

precision, recall, ap, preds_sorted, tp, fp = compute_ap(predictions, ground_truths)

print(f"{'rank':>4s}  {'conf':>5s}  {'TP':>3s}  {'FP':>3s}  {'cum-prec':>9s}  {'cum-rec':>8s}")
print("-" * 50)
for i, ((_, c), t, f, p, r) in enumerate(zip(preds_sorted, tp, fp, precision, recall)):
    print(f"{i+1:>4d}  {c:>5.2f}  {int(t):>3d}  {int(f):>3d}  {p:>9.3f}  {r:>8.3f}")
print(f"\nAP @ IoU=0.5  =  {ap:.4f}")


In [ ]:
# Plot the precision-recall curve.
plt.figure(figsize=(6, 5))
plt.plot(recall, precision, "o-", color="steelblue")
plt.fill_between(recall, 0, precision, alpha=0.2, color="steelblue")
plt.xlabel("recall"); plt.ylabel("precision")
plt.xlim(0, 1.05); plt.ylim(0, 1.05); plt.grid(alpha=0.3)
plt.title(f"Precision-Recall curve  |  AP = {ap:.3f}")
plt.show()
print("AP is roughly the area under this curve. Higher = better detector.")
print("mAP = mean of AP across all classes. (Here we only have one class.)")


### TODO 9.1
A perfect detector achieves **AP = 1.0**. A random detector approaches **AP ≈ 0**.
Verify both:

1. Construct a "perfect" set of predictions: one prediction per ground truth,
   each one exactly matching (same box), all with confidence 1.0. Compute the AP.
2. Construct a "random/bad" set: same number of predictions, all in wrong
   locations (no overlap with any GT). Compute the AP.


In [ ]:
# TODO 9.1 -- AP at the extremes.

# 1) A PERFECT detector: exactly the 3 GT boxes, confidence 1.0.
perfect = ???

# 2) A USELESS detector: 3 predictions, none overlapping any GT.
useless = ???

_, _, ap_perfect, _, _, _ = compute_ap(perfect, ground_truths)
_, _, ap_useless, _, _, _ = compute_ap(useless, ground_truths)
print(f"AP of a perfect detector : {ap_perfect:.4f}  (should be ~1.0)")
print(f"AP of a useless detector : {ap_useless:.4f}  (should be ~0.0)")


## 10 · Face recognition: Siamese network with triplet loss

> *Lecture recap:* face recognition has a **one-shot** problem — you might
> have only one photo per employee. So instead of training a softmax over
> identities (which fails with one image per class and breaks the moment a new
> employee is hired), we learn an **encoding function** `f(image) → vector` so
> that:
> - **same person** → encodings are close (small distance)
> - **different people** → encodings are far apart (large distance)
> A **Siamese network** is two copies of the same encoder; we compare the two
> output vectors.
> **Triplet loss** trains this: given an *Anchor*, a *Positive* (same person),
> and a *Negative* (different person):
> $$ L = \max(0,\ \|f(A) - f(P)\|^2 - \|f(A) - f(N)\|^2 + \alpha) $$
> where α is a **margin** (e.g. 0.2) preventing the trivial `f ≡ 0` solution.

To make this concrete and runnable in a lab, we'll train it on **MNIST digits
as identities**: two images of the **same digit** play the role of "same
person"; two images of **different digits** play "different people". Same
algorithm, no face dataset to download. The training takes ~1 minute on GPU.


In [ ]:
from torchvision import datasets

# Tiny MNIST -- use the standard download path.
mnist_tf = transforms.Compose([transforms.ToTensor()])
train_mnist = datasets.MNIST("./mnist_data", train=True,  download=True, transform=mnist_tf)
test_mnist  = datasets.MNIST("./mnist_data", train=False, download=True, transform=mnist_tf)
print("MNIST train:", len(train_mnist), " test:", len(test_mnist))

# Group images by their label so we can sample positives/negatives.
from collections import defaultdict
def index_by_label(ds):
    out = defaultdict(list)
    for i, (_, y) in enumerate(ds):
        out[y].append(i)
    return out

train_by_class = index_by_label(train_mnist)
test_by_class  = index_by_label(test_mnist)
print("examples per digit (train):", {k: len(v) for k, v in sorted(train_by_class.items())})


In [ ]:
# A Dataset that yields TRIPLETS (anchor, positive, negative).
class TripletMNIST(torch.utils.data.Dataset):
    def __init__(self, base_ds, by_class, length=20000):
        self.base = base_ds
        self.by_class = by_class
        self.classes = list(by_class.keys())
        self.length = length

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # 1) pick an anchor class, then sample two indices from that class
        anchor_cls = np.random.choice(self.classes)
        a_idx, p_idx = np.random.choice(self.by_class[anchor_cls], size=2, replace=False)
        # 2) pick a DIFFERENT class for the negative
        neg_cls = np.random.choice([c for c in self.classes if c != anchor_cls])
        n_idx = np.random.choice(self.by_class[neg_cls])
        return self.base[a_idx][0], self.base[p_idx][0], self.base[n_idx][0]

train_triplets = TripletMNIST(train_mnist, train_by_class, length=20000)
train_loader   = torch.utils.data.DataLoader(train_triplets, batch_size=128, shuffle=True)

# Show one triplet so the data structure is unambiguous.
a, p, n = train_triplets[0]
fig, axes = plt.subplots(1, 3, figsize=(7, 3))
for ax, im, t in zip(axes, [a, p, n], ["anchor", "positive (same)", "negative (different)"]):
    ax.imshow(im.squeeze(), cmap="gray"); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# The encoder f: a tiny conv net that outputs a 32-dim embedding vector.
class Encoder(nn.Module):
    def __init__(self, embed_dim=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 28 -> 14
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 14 -> 7
        )
        self.fc = nn.Linear(64 * 7 * 7, embed_dim)

    def forward(self, x):
        z = self.conv(x).flatten(1)
        z = self.fc(z)
        # L2-normalize so all embeddings live on the unit sphere -- standard
        # practice; makes distances bounded and well-behaved.
        return F.normalize(z, p=2, dim=1)

encoder = Encoder(embed_dim=32).to(device)
print(encoder)
print(f"trainable params: {sum(p.numel() for p in encoder.parameters()):,}")


In [ ]:
# The triplet loss -- exactly the lecture's formula.
def triplet_loss(anchor_emb, pos_emb, neg_emb, margin=0.2):
    d_ap = (anchor_emb - pos_emb).pow(2).sum(dim=1)    # ||f(A) - f(P)||^2  per item
    d_an = (anchor_emb - neg_emb).pow(2).sum(dim=1)    # ||f(A) - f(N)||^2  per item
    losses = F.relu(d_ap - d_an + margin)              # max(0, ...)
    return losses.mean()

# Train. ~1 minute on a Colab GPU; a few minutes on CPU.
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)
history = []
encoder.train()
EPOCHS = 2
for epoch in range(1, EPOCHS + 1):
    epoch_losses = []
    for a, p, n in train_loader:
        a, p, n = a.to(device), p.to(device), n.to(device)
        za, zp, zn = encoder(a), encoder(p), encoder(n)
        loss = triplet_loss(za, zp, zn, margin=0.2)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_losses.append(loss.item())
    history.extend(epoch_losses)
    print(f"epoch {epoch}/{EPOCHS}  mean batch loss: {np.mean(epoch_losses):.4f}")

plt.figure(figsize=(8, 3.5))
plt.plot(history, alpha=0.7)
plt.xlabel("batch"); plt.ylabel("triplet loss"); plt.title("training loss")
plt.grid(alpha=0.3); plt.show()
print("Loss dropping toward 0 = the network is learning to separate identities.")


In [ ]:
# Now verify it actually works the way the lecture promised: distances are
# small for same-class pairs and large for different-class pairs.
encoder.eval()
def embed(images):
    with torch.no_grad():
        return encoder(images.to(device)).cpu()

# Sample a bunch of same-class and different-class pairs from the TEST set.
np.random.seed(0)
same_dists, diff_dists = [], []
test_classes = list(test_by_class.keys())
for _ in range(500):
    c = np.random.choice(test_classes)
    i1, i2 = np.random.choice(test_by_class[c], size=2, replace=False)
    im1, im2 = test_mnist[i1][0], test_mnist[i2][0]
    e1, e2 = embed(torch.stack([im1, im2]))
    same_dists.append(float((e1 - e2).pow(2).sum().sqrt()))

    c2 = np.random.choice([k for k in test_classes if k != c])
    i1 = np.random.choice(test_by_class[c])
    i2 = np.random.choice(test_by_class[c2])
    im1, im2 = test_mnist[i1][0], test_mnist[i2][0]
    e1, e2 = embed(torch.stack([im1, im2]))
    diff_dists.append(float((e1 - e2).pow(2).sum().sqrt()))

print(f"same-class distance:   mean {np.mean(same_dists):.3f}, std {np.std(same_dists):.3f}")
print(f"diff-class distance:   mean {np.mean(diff_dists):.3f}, std {np.std(diff_dists):.3f}")
print("If learning worked, same << diff. The gap is exactly the embedding's discriminative power.")

# Visualize the two distributions.
plt.figure(figsize=(8, 4))
plt.hist(same_dists, bins=30, alpha=0.6, label="same identity (close)", color="seagreen")
plt.hist(diff_dists, bins=30, alpha=0.6, label="different identity (far)", color="crimson")
plt.xlabel("embedding distance"); plt.ylabel("count")
plt.title("distance between embeddings -- learned separation")
plt.legend(); plt.grid(alpha=0.3); plt.show()


## 11 · Face verification as binary classification — the alternative head

> *Lecture recap:* an alternative to triplet loss is to **build a binary
> classifier on top of the Siamese encoder**: feed both images through the same
> encoder to get `f(x1)` and `f(x2)`, take the element-wise `|f(x1) - f(x2)|`
> (or chi-squared form), pass through a logistic regression layer, predict 1 if
> same person, 0 otherwise. Same encoder, different training objective.

The key practical insight from the lecture: at deployment, you can
**pre-compute the encoding of every employee in your database**. When someone
walks up to the door, you only compute *their* encoding and compare to the
pre-computed bank. That's a big efficiency win.

Let's demonstrate this with the encoder we just trained, *without* training a
separate logistic head — just use the distance directly with a threshold (the
practical "verify" check that the lecture's `d(x1, x2) < τ` describes).


In [ ]:
# Pre-compute the "employee database" embeddings -- in practice this would be
# one embedding per registered employee. We use one image per digit class.
encoder.eval()
database = {}                          # digit_class -> 32-dim embedding
for cls, idxs in test_by_class.items():
    img, _ = test_mnist[idxs[0]]       # one image per class
    database[cls] = embed(img.unsqueeze(0))[0]
print(f"Database built with {len(database)} 'identities' (digit classes).")

def verify(probe_img, database, tau=0.5):
    """
    Given a probe image, find the closest database entry. Return (predicted_id,
    distance, is_match). It's a 'match' only if distance < tau.
    """
    probe_emb = embed(probe_img.unsqueeze(0))[0]
    best_cls, best_d = None, float("inf")
    for cls, emb in database.items():
        d = float((probe_emb - emb).pow(2).sum().sqrt())
        if d < best_d:
            best_d, best_cls = d, cls
    return best_cls, best_d, best_d < tau

# Test it.
correct = 0
n = 200
for _ in range(n):
    cls = np.random.choice(list(test_by_class.keys()))
    idx = np.random.choice(test_by_class[cls])
    img, true_cls = test_mnist[idx]
    pred, d, matched = verify(img, database, tau=0.8)
    if matched and pred == true_cls:
        correct += 1
print(f"\n1-shot 'recognition' accuracy on {n} probes: {correct/n:.2%}")
print("(Each probe is compared to a database with ONE image per class.")
print(" This is exactly the lecture's one-shot scenario.)")


### TODO 11.1
Build a quick **τ-sweep** to make the threshold's effect concrete. Use the
`verify` function and a fixed test sample. For each τ in
`[0.2, 0.5, 0.8, 1.1, 1.4]`, compute:
- **TAR** (True Acceptance Rate): fraction of *same-identity* probes correctly
  accepted as the right person.
- **FAR** (False Acceptance Rate): fraction of *impostor* probes wrongly accepted.

You want **TAR high** and **FAR low**. The "best" τ depends on the cost
trade-off for your application.


In [ ]:
# TODO 11.1 -- sweep tau to see the TAR/FAR trade-off.
np.random.seed(1)
n_per = 200

# Build genuine (same-identity) and impostor (different-identity) probes.
genuine, impostor = [], []
for _ in range(n_per):
    cls = np.random.choice(list(test_by_class.keys()))
    idx = np.random.choice(test_by_class[cls])
    img, _ = test_mnist[idx]
    genuine.append((img, cls))               # probe SHOULD match class `cls`

    cls_true = np.random.choice(list(test_by_class.keys()))
    cls_fake = np.random.choice([c for c in test_by_class if c != cls_true])
    idx = np.random.choice(test_by_class[cls_true])
    img, _ = test_mnist[idx]
    impostor.append((img, cls_fake))         # probe should NOT match class `cls_fake`

print(f"{'tau':>5s}  | {'TAR':>6s}  {'FAR':>6s}  comment")
print("-" * 45)
for tau in [0.2, 0.5, 0.8, 1.1, 1.4]:
    # TAR: of genuine probes, fraction where verify(...) accepts AND identifies correctly
    tar = ???
    # FAR: of impostor probes, fraction where verify(...) accepts as the claimed class
    far = ???
    print(f"{tau:>5.1f}  | {tar:>6.2%}  {far:>6.2%}")
